# 05 統合 — AI が見つけ、OpenCV が処理する

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 解説（この文章）→ コード → 結果（画像）→ ✅ チェックポイント、の順で進みます。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

> ### ⚠️ 先に AI カメラのモデルを入れてください
> このノートは **AI カメラ用モデル `imx500-all`** が必要です。**Pi のターミナル**（SSH）で一度だけ実行します（約1分・オンライン）:
>
> ```bash
> sudo apt install -y imx500-all
> ```
>
> - もし `Firmware file /usr/share/imx500-models/....rpk does not exist.` と出たら、**これが未導入**という意味です。上のコマンドで入れてから、このノートを上から実行し直してください。
> - この Pi は**再起動すると配布時の状態に戻る**ため、再起動後はもう一度このコマンドが必要です。

**認識は AI、後処理は OpenCV** という実務の型を、小さなアプリで体験します。

> AI カメラが「何があるか」を検出し、OpenCV が「切り出し・ぼかし・集計」といった後処理を担当する**役割分担**。
> 例：人数カウンタ／在・不在ログ（第1回のデータ保存の応用）／検出した人の領域だけぼかすプライバシー配慮。

## 5-1. 準備（AIカメラ起動）

In [ ]:
import cv2, time
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
%matplotlib inline
from picamera2 import Picamera2
from picamera2.devices import IMX500
MODEL = '/usr/share/imx500-models/imx500_network_ssd_mobilenetv2_fpnlite_320x320_pp.rpk'
imx = IMX500(MODEL); labels = imx.network_intrinsics.labels
picam = Picamera2(imx.camera_num)
picam.configure(picam.create_preview_configuration(
    main={'size':(640,480),'format':'RGB888'}, buffer_count=8))
imx.show_network_fw_progress_bar(); picam.start(); time.sleep(2)
print('準備OK')

## 5-2. 検出した物体を切り出して保存
AI が見つけた領域を OpenCV のスライスで切り出し、`crop_<ラベル>_n.jpg` に保存します。
（**認識=AI、切り出し=OpenCV** の最小の組み合わせ）

In [ ]:
saved = []
for _ in range(30):
    md_ = picam.capture_metadata()
    outputs = imx.get_outputs(md_, add_batch=True)
    frame = picam.capture_array()
    if outputs is None: continue
    boxes, scores, classes = outputs[0][0], outputs[1][0], outputs[2][0]
    for box, score, cls in zip(boxes, scores, classes):
        if score < 0.5: continue
        x,y,w,h = imx.convert_inference_coords(box, md_, picam)
        x,y = max(0,x), max(0,y)
        crop = frame[y:y+h, x:x+w]
        if crop.size == 0: continue
        name = labels[int(cls)]
        fn = f'crop_{name}_{len(saved)}.jpg'
        cv2.imwrite(fn, crop); saved.append((fn, crop))
    break
print('切り出し:', [s[0] for s in saved])
for fn, crop in saved[:6]:
    plt.figure(figsize=(3,3)); plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title(fn); plt.show()

## 5-3. 発展課題（1 つ選ぶ）
- **人数カウンタ**：`person` の検出数を数えて画面に表示
- **在/不在ログ**：`person` を検出した時刻だけ `log.csv` に追記（第1回の応用）
- **プライバシー配慮**：検出した `person` の領域だけ `cv2.GaussianBlur` でぼかす
- **色 + AI**：`bottle` を検出し、その領域を HSV で色判定

> 🤖 **ChatGPT に聞いてみよう**
> 選んだ課題を ChatGPT に丸ごと相談しよう。例：
「IMX500 の検出結果(boxes, scores, classes)が手元にある。`person` だけ数えて画像左上に『人数: N』と描くコードを書いて。OpenCV は BGR」。
**出たコードはまず動かして確認**。エラーは全文を貼って直してもらう。

In [ ]:
# ここに自分のアプリを書く（ChatGPT に相談しながらでOK）

## 5-9. 後始末（必ず実行）

In [ ]:
picam.close(); print('お疲れさまでした！')